# 拓展：强制使用工具
## tool_choice 参数说明
`bind_tools` 可以传递参数 `tool_choice` ，用于控制是否强制使用工具。
该字段最终会作为 `payload` 的 `tool_choice` 字段传递给模型，OpenAI和Deepseek的官方API服务对于 tool_choice 的取值做了相同的规定。
- none ：模型不会调用任何工具。
- auto ： 默认值 ，模型可以自主决定不调用或调用任意数量的工具。
- required ：模型必须调用工具，数量不限。
tool_choice 还支持传递 any ，等价于 required 。

In [2]:
from dotenv import load_dotenv
import os
from langchain_qwq import ChatQwen
from langchain.tools import tool
from langchain.messages import HumanMessage

load_dotenv(override=True)

model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'
model_with_tools = model.bind_tools([get_weather],tool_choice="none")

messages = [
    HumanMessage("今天北京天气如何？别瞎编")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

抱歉，我无法直接获取实时天气数据。建议你打开联网搜索功能，或者查看手机上的天气应用、访问中国天气网等权威渠道，获取今天北京最新的准确天气信息。


## 写在最后
### 如何处理工具调用失败问题？
1. 工具内部处理（try： except Exception as e:）
2. Agent级重试（使用prompt）
3. 调用级重试（@retry(stop=stop_after_attempt(3)))）

### 工具尽量返回字符串
在编写传统的 Python 代码时，返回字典（ dict ）显然更方便后续代码处理。但在 LangChain 的工具（Tools）生态中，强烈建议工具返回字符串（str）。因为：
1. 大模型（LLM）的本质只吃“文本”
2. 避免大模型“胡思乱想”（乱码与格式问题）

### 选择同步 vs 异步
同步工具 ：简单场景，CPU 密集型任务
异步工具 ：IO 密集型（API 调用、数据库、文件操作）